In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [1]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

#My new imports (just to keep track)
from qiskit_aer import AerSimulator
import numpy as np

theta = 2 * np.arccos(np.sqrt(1/3))

def quantum_choice(sim):

    qc = QuantumCircuit(1,1)
    qc.ry(theta,0)
    qc.measure(0,0)

    result = sim.run(qc, shots=1).result()
    bit = int(list(result.get_counts().keys())[0])

    if bit == 0:
        return 1

    qc2 = QuantumCircuit(1,1)
    qc2.h(0)
    qc2.measure(0,0)

    result2 = sim.run(qc2, shots=1).result()
    bit2 = int(list(result2.get_counts().keys())[0])

    if bit2 == 0:
        return 2
    else:
        return 3

def measure_w(qc, qubit, classical_bit):
    qc.ry(-np.pi/4, qubit)
    qc.measure(qubit, classical_bit)

def measure_v(qc, qubit, classical_bit):
    qc.ry(np.pi/4, qubit)
    qc.measure(qubit, classical_bit)

def measure_z(qc, qubit, classic_bit):
    qc.measure(qubit, classic_bit)

def measure_x(qc, qubit, classic_bit):
    qc.h(qubit)
    qc.measure(qubit, classic_bit)

def eve_intercept(qc):
    # Eve measures Bob's qubit in the Z basis
    qc.measure(1, 1)
    

def create_bell_pair():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.x(1)
    qc.z(0)
    return qc

def bit_to_pm1(bit):
    return 1 if bit == '0' else -1

sim = AerSimulator()

def expectation_value(measure_A, measure_B, shots=2000):

    qc = create_bell_pair()

    measure_A(qc, 0, 0)
    measure_B(qc, 1, 1)

    qc = qc.reverse_bits()
    result = sim.run(qc, shots=shots).result()
    counts = result.get_counts()

    total = 0

    for bitstring, count in counts.items():
        a = bit_to_pm1(bitstring[0])
        b = bit_to_pm1(bitstring[1])
        total += a * b * count

    return total / shots

#Test needed terms
E_XW = expectation_value(measure_x, measure_w)
E_XV = expectation_value(measure_x, measure_v)
E_ZW = expectation_value(measure_z, measure_w)
E_ZV = expectation_value(measure_z, measure_v)

S = abs(E_XW - E_XV + E_ZW + E_ZV)

print("S =", S)


import random

def random_A():
    return quantum_choice(sim)

def random_B():
    return quantum_choice(sim)

A_map = {
    1: measure_x,
    2: measure_w,
    3: measure_z
}

B_map = {
    1: measure_w,
    2: measure_z,
    3: measure_v
}

def run_round():

    i = random_A()
    j = random_B()

    qc = create_bell_pair()

    # Eve intercepts Bob's qubit before Bob measures
    qc.measure(1,1)

    A_map[i](qc, 0, 0)
    B_map[j](qc, 1, 1)

    qc = qc.reverse_bits()
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()

    bitstring = list(counts.keys())[0]

    return i, j, bitstring

#Run the simulation itself
N = 500
rounds = int(9*N/2)

results = [run_round() for _ in range(rounds)]

alice_key = []
bob_key = []

#Keep appropriate rounds
for i, j, bits in results:
    if (i == 2 and j == 1) or (i == 3 and j == 2):

        alice_bit = bits[0]
        bob_bit = bits[1]

        bob_bit = '0' if bob_bit == '1' else '1'  # Bob inverts

        alice_key.append(alice_bit)
        bob_key.append(bob_bit)

print("Keys match:", alice_key == bob_key)
print("Key length:", len(alice_key))


def compute_S_from_results(results):

    sums = {
        (1,1): 0,
        (1,3): 0,
        (3,1): 0,
        (3,3): 0
    }

    counts = {
        (1,1): 0,
        (1,3): 0,
        (3,1): 0,
        (3,3): 0
    }

    for i, j, bits in results:

        if (i, j) in sums:

            a = 1 if bits[0] == '0' else -1
            b = 1 if bits[1] == '0' else -1

            sums[(i,j)] += a * b
            counts[(i,j)] += 1

    E_XW = sums[(1,1)] / counts[(1,1)]
    E_XV = sums[(1,3)] / counts[(1,3)]
    E_ZW = sums[(3,1)] / counts[(3,1)]
    E_ZV = sums[(3,3)] / counts[(3,3)]

    S = abs(E_XW - E_XV + E_ZW + E_ZV)

    return S

S_protocol = compute_S_from_results(results)
print("S from protocol data =", S_protocol)

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

S = 2.8200000000000003
Keys match: False
Key length: 512
S from protocol data = 1.3212717196922095
